# Setup & Daten Laden

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import numpy as np
from tqdm import tqdm

from data import load_data, prepare_splits, NUM_CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch nutzt Device: {device}")

# Pfade anpassen!
DATA_PATH = "./../data/Galaxy10_DECals.h5"
# Trage hier den exakten Dateinamen deines trainierten Zoobot-Modells ein
ZOOBOT_CHECKPOINT = "./../reports/zoobot/zoobot_dense_only_frozen_20260606_133047_746986_model.pth" 
BATCH_SIZE = 32

print("Lade Rohbilder...")
images_raw, labels = load_data(DATA_PATH)

# WICHTIG: random_state=42 garantiert denselben Split wie später in Keras
split_folds, X_test, y_test = prepare_splits(
    labels, use_kfold=False, val_size=0.2, test_size=0.2, random_state=42
)
train_idx, val_idx = split_folds[0]
print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(X_test)}")

# Model Laden

In [ ]:
class ZoobotTeacher(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = timm.create_model("hf_hub:mwalmsley/zoobot-encoder-convnext_nano", pretrained=False, num_classes=0)
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 256),
            nn.GELU(),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))

teacher_model = ZoobotTeacher(num_classes=NUM_CLASSES).to(device)
teacher_model.load_state_dict(torch.load(ZOOBOT_CHECKPOINT, map_location=device))
teacher_model.eval()
print("Teacher-Modell erfolgreich geladen!")

# Inference & Export

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, images_array, indices):
        self.images = images_array
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        img_np = self.images[real_idx]
        img_np = np.transpose(img_np, (2, 0, 1)) # HWC zu CHW
        return torch.tensor(img_np, dtype=torch.float32) / 255.0

def extract_logits(indices, desc):
    dataset = InferenceDataset(images_raw, indices)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    all_logits = []
    with torch.no_grad():
        for inputs in tqdm(loader, desc=desc):
            logits = teacher_model(inputs.to(device))
            all_logits.append(logits.cpu().numpy())
    return np.concatenate(all_logits, axis=0)

print("Extrahiere Logits...")
train_logits = extract_logits(train_idx, "Train Logits")
val_logits = extract_logits(val_idx, "Val Logits")

np.save("teacher_train_logits.npy", train_logits)
np.save("teacher_val_logits.npy", val_logits)
print("Fertig! Logits als .npy gespeichert. Du kannst dieses Notebook nun schließen.")